In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('DB/youtube.csv')

print(df.count())
display(df.head())

link           3599
title          3599
description    3599
category       3599
dtype: int64


,link,title,description,category
0,JLZlCZ0,Ep 1| Travelling through North East India | Of...,Tanya Khanijow\r\n671K subscribers\r\nSUBSCRIB...,travel
1,i9E_Blai8vk,Welcome to Bali | Travel Vlog | Priscilla Lee,Priscilla Lee\r\n45.6K subscribers\r\nSUBSCRIB...,travel
2,r284c-q8oY,My Solo Trip to ALASKA | Cruising From Vancouv...,Allison Anderson\r\n588K subscribers\r\nSUBSCR...,travel
3,Qmi-Xwq-ME,Traveling to the Happiest Country in the World!!,Yes Theory\r\n6.65M subscribers\r\nSUBSCRIBE\r...,travel
4,_lcOX55Ef70,Solo in Paro Bhutan | Tiger's Nest visit | Bhu...,Tanya Khanijow\r\n671K subscribers\r\nSUBSCRIB...,travel


In [4]:
df.drop(columns=['link'], inplace=True)

# Analisis Exploratorio

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3599 entries, 0 to 3598
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        3599 non-null   object
 1   description  3599 non-null   object
 2   category     3599 non-null   object
dtypes: object(3)
memory usage: 84.5+ KB


## Limpieza

Podemos ver que no tenemos datos nulos

In [6]:
df_clean = df.copy()
df_clean.drop_duplicates(inplace=True)

In [7]:
df_clean.count()

title          3433
description    3433
category       3433
dtype: int64

In [8]:
df["title"] = df["title"].astype(str).str.strip()
df["description"] = df["description"].astype(str).fillna("").str.strip()

In [9]:
print("Categorías únicas:")
print(df_clean['category'].unique())

Categorías únicas:
['travel' 'food' 'art_music' 'history']


In [10]:
travel_count = df_clean[df_clean['category'] == 'travel'].shape[0]
print("travel:", travel_count)

food_count = df_clean[df_clean['category'] == 'food'].shape[0]
print("food:", food_count)

art_music_count = df_clean[df_clean['category'] == 'art_music'].shape[0]
print("art_music:", art_music_count)

history_count = df_clean[df_clean['category'] == 'history'].shape[0]
print("history:", history_count)

travel: 1127
food: 887
art_music: 831
history: 588


Como podemos ver nuestras categorias no estan balanceadas, siendo mas las de 'travel' que son casi el doble de las de 'history'

Longitud de caracteres por titulo y descripcion

In [11]:
title_lengths = df_clean['title'].str.len()
print("Titulo")
print("Promedio:", title_lengths.mean())
print("Mínimo:", title_lengths.min())
print("Máximo:", title_lengths.max())


Titulo
Promedio: 67.04340227206525
Mínimo: 6
Máximo: 101


In [12]:
title_lengths = df_clean['description'].str.len()
print("Descripción")
print("Promedio:", title_lengths.mean())
print("Mínimo:", title_lengths.min())
print("Máximo:", title_lengths.max())

Descripción
Promedio: 426.7707544421788
Mínimo: 23
Máximo: 5239


# Modelo

In [25]:
from datasets import Dataset, ClassLabel
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
import evaluate

## Separar (train/val/test)

In [16]:
classes = sorted(df["category"].astype(str).unique().tolist())
label2id = {c:i for i, c in enumerate(classes)}
id2label = {i:c for c, i in label2id.items()}

# Crear columna 'label' numérica a partir de 'category'
df = df.copy()
df["label"] = df["category"].astype(str).map(label2id)

# Split estratificado
train_df, val_df = train_test_split(
    df[["title","label"]],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)

### RoBERTa

In [18]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess(batch):
    return tokenizer(
        batch["title"], truncation=True, padding=False, max_length=MAX_LEN
    )

train_ds = train_ds.map(preprocess, batched=True)
val_ds   = val_ds.map(preprocess,   batched=True)

# Trainer espera la columna 'labels'
train_ds = train_ds.rename_columns({"label": "labels"})
val_ds   = val_ds.rename_columns({"label": "labels"})

data_collator = DataCollatorWithPadding(tokenizer)


Map: 100%|██████████| 720/720 [00:00<00:00, 38885.66 examples/s]


### Modelo

In [19]:
num_labels = len(classes)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Entrenamiento/ Evaluacion

In [28]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

args = TrainingArguments(
    output_dir="roberta-cls",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",          # <-- antes: evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
print(trainer.evaluate())
trainer.save_model("roberta-cls/best")
tokenizer.save_pretrained("roberta-cls/best")

C:\Users\death\AppData\Local\Temp\ipykernel_6212\2039786274.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.134330,0.970833,0.971335
2,No log,0.141332,0.968056,0.969174
3,0.187000,0.129372,0.973611,0.974409


C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\death\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.1293715536594391, 'eval_accuracy': 0.9736111111111111, 'eval_f1_macro': 0.9744092533755377, 'eval_runtime': 25.9157, 'eval_samples_per_second': 27.782, 'eval_steps_per_second': 1.736, 'epoch': 3.0}


('roberta-cls/best\\tokenizer_config.json',
 'roberta-cls/best\\special_tokens_map.json',
 'roberta-cls/best\\vocab.json',
 'roberta-cls/best\\merges.txt',
 'roberta-cls/best\\added_tokens.json',
 'roberta-cls/best\\tokenizer.json')

In [30]:
import evaluate
import torch

def predict(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():
        out = model(**{k: v.to(model.device) for k, v in enc.items()})
        pred_ids = out.logits.argmax(dim=-1).cpu().numpy().tolist()
    return [model.config.id2label[i] for i in pred_ids]

print(predict([
    "I loved the museum and the old town tour.",
    "The tacos were amazing and fresh!",
    "El concierto de anoche fue espectacular, la banda tocó todos sus éxitos y el público estaba encantado."
]))

['history', 'food', 'art_music']
